# 02 - LangGraph Guardrails

This notebook builds and runs several of the LangGraph workflows in `src/guardrails_demo/langgraph_guardrails/workflows.py`. Each graph composes the same LangChain-layer guardrail functions from notebook 01, adding state, branching, retries, and approval.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

## Basic input guardrail flow

`START -> validate_input -> agent -> END`, with a rejection branch.

In [2]:
from guardrails_demo.langgraph_guardrails.state import new_state
from guardrails_demo.langgraph_guardrails.workflows import build_basic_input_guardrail_graph

graph = build_basic_input_guardrail_graph()
for text in ['What is a Python list?', '']:
    result = graph.invoke(new_state(text))
    print(repr(text), '->', result['guardrail_status'], '-', result['output'])

'What is a Python list?' -> allowed - [stub-response] Here is a placeholder answer for: What is a Python list?
'' -> blocked - Request blocked: Input is empty or whitespace only.


## Multi-guardrail pipeline

input -> PII -> topic -> safety -> agent -> output, each gated.

In [3]:
from guardrails_demo.langgraph_guardrails.workflows import build_multi_guardrail_pipeline_graph

graph = build_multi_guardrail_pipeline_graph()
for text in ['What is a Python function?', "Tell me today's cricket score."]:
    result = graph.invoke(new_state(text))
    print(text, '->', result['guardrail_status'])

What is a Python function? -> allowed
Tell me today's cricket score. -> blocked


## Retry and repair

A bounded loop between `agent` and `validate_output`, capped by `MAX_RETRIES`.

In [4]:
from guardrails_demo.langgraph_guardrails.workflows import build_retry_and_repair_graph

graph = build_retry_and_repair_graph()
result = graph.invoke(new_state('What is a Python list?'))
print('attempt_count:', result['attempt_count'])
print('output:', result['output'])

attempt_count: 0
output: [stub-response] Here is a placeholder answer for: What is a Python list?


## Human approval

A mock sensitive action gated by a simulated reviewer.

In [5]:
from guardrails_demo.langgraph_guardrails.workflows import build_human_approval_graph

graph = build_human_approval_graph()
result = graph.invoke(new_state('What is a Python list?'))
print(result['output'])

[stub-response] Here is a placeholder answer for: What is a Python list?


## Tool execution guardrail

Deny-by-default: any tool not on the allowlist is rejected.

In [6]:
from guardrails_demo.langgraph_guardrails.workflows import build_tool_execution_guardrail_graph

graph = build_tool_execution_guardrail_graph()
state = new_state('calculate')
state['tool_name'] = 'calculator'
state['tool_arguments'] = {'operation': 'add', 'left': 2, 'right': 3}
result = graph.invoke(state)
print(result['output'])

5.0


## Agent loop protection

A hard ceiling on tool-calling iterations prevents a runaway agent.

In [7]:
from guardrails_demo.langgraph_guardrails.workflows import build_agent_loop_protection_graph

graph = build_agent_loop_protection_graph()
state = new_state('loop')
state['requested_tool_calls'] = ['calculator'] * 10
state['tool_name'] = 'calculator'
state['tool_arguments'] = {'operation': 'add', 'left': 1, 'right': 1}
result = graph.invoke(state)
print('tool_call_count:', result['tool_call_count'], '| escalation_required:', result['escalation_required'])

tool_call_count: 3 | escalation_required: True


## Next

See `03_agent_security_guardrails.ipynb` for prompt injection protection and the full production-style agent.